In [295]:
import numpy as np

def pk_get_nh_idx(img, indices):
    """
    Python translation of MATLAB pk_get_nh_idx.
    - img: 3D numpy array (shape (W,H,D)). Should be padded externally (like MATLAB padarray).
    - indices: 1D array-like of linear indices (numpy 0-based indices into img.ravel()).
    Returns:
      nhood: (N,27) int64 array. Each row contains 27 0-based linear indices for that voxel's 3x3x3 neighborhood.
    """
    indices = np.asarray(indices, dtype=np.int64)
    if indices.ndim != 1:
        indices = indices.ravel()

    w, h, d = img.shape
    # unravel to coordinates (0-based)
    x, y, z = np.unravel_index(indices, (w, h, d))
    # prepare output
    n = len(indices)
    nhood = np.empty((n, 27), dtype=np.int64)

    col = 0
    # MATLAB loops xx=1:3,yy=1:3,zz=1:3 and uses x+xx-2 etc.
    # We loop offsets (-1,0,+1) which corresponds to xx-2 in MATLAB.
    for dx in (-1, 0, 1):
        for dy in (-1, 0, 1):
            for dz in (-1, 0, 1):
                # neighbor coordinates
                nx = x + dx
                ny = y + dy
                nz = z + dz
                # convert to linear index (0-based)
                nhood[:, col] = np.ravel_multi_index((nx, ny, nz), (w, h, d))
                col += 1

    return nhood

def pk_get_nh(img, indices):
    """
    Python translation of MATLAB pk_get_nh.
    - img: 3D numpy array (dtype can be bool or int). Should be padded externally.
    - indices: 1D array-like of linear indices (0-based).
    Returns:
      nhood: (N,27) boolean numpy array; True if voxel is foreground/non-zero.
    """
    indices = np.asarray(indices, dtype=np.int64).ravel()
    w, h, d = img.shape
    x, y, z = np.unravel_index(indices, (w, h, d))

    n = len(indices)
    nhood = np.empty((n, 27), dtype=bool)

    col = 0
    for dx in (-1, 0, 1):
        for dy in (-1, 0, 1):
            for dz in (-1, 0, 1):
                nx = x + dx
                ny = y + dy
                nz = z + dz
                linear_idx = np.ravel_multi_index((nx, ny, nz), (w, h, d))
                # pick values from img by flat indexing
                nhood[:, col] = img.ravel()[linear_idx].astype(bool)
                col += 1

    return nhood

def pk_follow_link(skel2, nodes, curr_node_cluster_idx, curr_node_idx, can_idx, cans, c2n):
    '''
    Returns the the path,  from the current node and its current cannal voxel neighbor to
    another node (endpoint or not)

    Inupts: 
        - skel2 [np.ndarray]: skeletonized image with identified node clusters (clusters have unique id [2, inf], all 1s are
            cannal voxels, and all 0s are background voxels)
        - nodes [list[node_cluster]]: list of all node_clusters in the network
        - 
    '''
    path = []
    ep = 0

    path.append(int(nodes[curr_node_cluster_idx].nodes_idx[curr_node_idx]))

    done = False
    while not done:
        next_cand = c2n[can_idx] - 1
        cand = cans[next_cand][1]

        # swtich to other cannal voxel neighbor
        if cand == path[-1]:
            cand = cans[next_cand][2]

        if skel2.flat[cand] > 1:
            path.append(can_idx)
            path.append(int(cand))
            n_idx = skel2.flat[cand] - 2

            ep = nodes[n_idx].ep
            done = True

        else:
            path.append(curr_node_idx)
            curr_node_idx = cand

    return (path, n_idx, ep)

# path = pk_follow_link(skel2, nodes, 0, 0, 22, cans, c2n)

In [296]:
class node_cluster:
    def __init__(self, nodes_idx: list[int], ep: int):
        self.nodes_idx = nodes_idx
        self.links = []
        self.conn = []
        self.ep = ep
        self.comx = None
        self.comy = None
        self.comz = None
        self.label = None

    def find_center_mass(self, skel_dim):
        x, y, z = np.unravel_index(self.nodes_idx, skel_dim)
        self.comx = np.mean(x)
        self.comy = np.mean(y)
        self.comz = np.mean(z)
    
    def assign_label(self, lm: np.ndarray):
        self.label = lm.ravel()[self.nodes_idx[0]]

In [297]:
# Create a 4x4 uint32 array of zeros
test_skel = np.array([
    [1, 0, 0, 0, 0, 0, 1],
    [0, 1, 0, 0, 0, 1, 0],
    [0, 0, 1, 1, 1, 0, 0],
    [0, 1, 0, 0, 0, 1, 0],
    [1, 0, 0, 0, 0, 0, 1]
])

skel2 = np.array([
    [2, 0, 0, 0, 0, 0, 3],
    [0, 1, 0, 0, 0, 1, 0],
    [0, 0, 4, 1, 5, 0, 0],
    [0, 1, 0, 0, 0, 1, 0],
    [6, 0, 0, 0, 0, 0, 7]
])

# Add a 3rd dimension (equivalent to test_skel(:, :, 1) in MATLAB)
test_skel = test_skel[:, :, np.newaxis]
skel2 = skel2[:, :, np.newaxis]

skel = np.pad(test_skel, pad_width=1, mode='constant', constant_values=0)
skel2 = np.pad(skel2, pad_width=1, mode='constant', constant_values=0)

list_canal = np.flatnonzero(skel)

nh = pk_get_nh(skel, list_canal)
nhi = pk_get_nh_idx(skel, list_canal)

sum_nh = np.sum(nh.astype(bool), axis=1)

nodes = list_canal[sum_nh > 3]
ep = list_canal[sum_nh == 2]
cans = list_canal[sum_nh == 3]

can_nh_idx = pk_get_nh_idx(skel, cans)
can_nh = pk_get_nh(skel, cans)

# remove center of 3x3x3 cube (position 14 in MATLAB → index 13 in Python)
can_nh_idx = np.delete(can_nh_idx, 13, axis=1)
can_nh = np.delete(can_nh, 13, axis=1)

can_nb = np.sort(can_nh * can_nh_idx, axis=1)
can_nb = can_nb[:, -2:]

cans = np.column_stack((cans, can_nb))

w, l, h = skel.shape  # same as in MATLAB
num_voxels = w * l * h

c2n = np.zeros((9, 7, 3), dtype=int)
c2n_flat = c2n.ravel()
c2n_flat[cans[:, 0]] = np.arange(1, cans.shape[0] + 1)
c2n = c2n_flat

# s2n: map each neighborhood center voxel (from nhi[:, 13]) to its row number in 'nhi'
s2n = np.zeros(num_voxels, dtype=int)
s2n[nhi[:, 13]] = np.arange(1, nhi.shape[0] + 1)

link_idx = s2n[[31]] - 1

link_cands = nhi[link_idx[0], nh[link_idx[0], :] == 1]

ep_cands = np.intersect1d(link_cands, ep)

# filter for cannal voxels connected to curr node (link candidates / possible paths)
coords = np.unravel_index(link_cands, skel2.shape)
mask = skel2[coords] == 1
link_cands = link_cands[mask]

link_cands = np.intersect1d(link_cands, cans[:, 0])

ep_cands = np.intersect1d(link_cands, ep)


# filter for cannal voxels connected to curr node (link candidates / possible paths)
coords = np.unravel_index(link_cands, skel2.shape)
mask = skel2[coords] == 1
link_cands = link_cands[mask]

link_cands = np.intersect1d(link_cands, cans[:, 0])

In [298]:
node_31 = node_cluster([31], 1)
node_49 = node_cluster([49], 1)
node_91 = node_cluster([91], 0)
node_97 = node_cluster([97], 0)
node_139 = node_cluster([139], 1)
node_157 = node_cluster([157], 1)

nodess = [node_31, node_49, node_91, node_97, node_139, node_157]

In [299]:
path, idx, ep = pk_follow_link(skel2, nodess, 3, 0, 73, cans, c2n)

In [300]:
path

[97, 73, 49]

In [301]:
x, y, z = np.unravel_index([97], (w, l, h))

In [302]:
print(y, z, z)

[5] [1] [1]


# FUCKING AROUND ZONE

In [ ]:
from scipy.ndimage import label
import numpy as np

skel = np.array([
    [1, 0, 0, 0, 0, 0, 1],
    [0, 1, 0, 0, 0, 1, 0],
    [0, 0, 1, 1, 1, 0, 0],
    [0, 1, 0, 0, 0, 1, 0],
    [1, 0, 0, 0, 0, 0, 1]
])

# pad volume with zeros
skel = skel[:, :, np.newaxis]
skel = np.pad(skel, pad_width=1, mode='constant')

# label connected components 
lm, num_components= label(skel, structure=np.ones((3, 3, 3)))

skel2 = skel.astype(np.uint16) # needed for labeling nodes etc.

# all foreground (True) voxels, as flat indices
list_canal = np.flatnonzero(skel)

# if image contains no skeletonized data, return empty elements
if len(list_canal) == 0:
    raise ValueError("No skeletonized data: the input 3D mask contains no foreground voxels.")

# 26-nh of all canal voxels
nh = pk_get_nh(skel, list_canal)       # shape: (len(list_canal), 27)

# 26-nh indices of all canal voxels
nhi = pk_get_nh_idx(skel, list_canal)  # shape: (len(list_canal), 27)

# num of 26-nb of each skel voxel + 1
sum_nh = np.sum(nh.astype(bool), axis=1)

# node/endpoint/canal assignment
nodes = list_canal[sum_nh > 3] # all canal voxels with >2 nb are nodes -> branching points
ep    = list_canal[sum_nh == 2] # all canal voxels with exactly one nb are end nodes -> endpoints
cans  = list_canal[sum_nh == 3] # all canal voxels with exactly 2 nb -> segments

# Nx3 matrix with the 2 neighbors of each canal voxel
if cans.size > 0:
    can_nh_idx = pk_get_nh_idx(skel, cans)
    can_nh = pk_get_nh(skel, cans)

    # remove center of 3x3 cube
    can_nh_idx = np.delete(can_nh_idx, 13, axis=1)
    can_nh = np.delete(can_nh, 13, axis=1)

    # keep only the two existing foreground voxels
    can_nb = np.sort(can_nh * can_nh_idx, axis=1)
    can_nb = can_nb[:, -2:]  # keep last two columns

    # add neighbours to canalicular voxel list (this might include nodes)
    cans = np.column_stack([cans, can_nb])
else:
    cans = np.empty((0, 3), dtype=int)

# group clusters of node voxels to nodes
node, link = [],[]

# add node metadata
tmp = np.zeros(skel.shape, dtype=bool)
tmp.ravel()[nodes] = True
labelimg, num_clustered_nodes = label(tmp, structure=np.ones((3,3,3))) # number of unique nodes
cluster_indices = [np.flatnonzero(labelimg == i) for i in range(1, num_clustered_nodes+1)]

for i in range(num_clustered_nodes):
    l = lm.flatten()[cluster_indices]
    new_node = node_cluster(cluster_indices[i], 0)
    new_node.find_center_mass(skel.shape)
    new_node.assign_label(lm)
    node.append(new_node)

    # assign index to node voxels in skel2
    id = i + 2
    skel2.ravel()[new_node.nodes_idx] = id

# add endpoint metadata
tmp = np.zeros(skel.shape, dtype=bool)
tmp.ravel()[ep] = True
labelimg, num_clustered_nodes = label(tmp, structure=np.ones((3,3,3))) # number of unique nodes
cluster_indices = [np.flatnonzero(labelimg == i) for i in range(1, num_clustered_nodes+1)]


for i in range(num_clustered_nodes):
    l = lm.flatten()[cluster_indices]
    new_node = node_cluster(cluster_indices[i], 0)
    new_node.find_center_mass(skel.shape)
    new_node.assign_label(lm)
    node.append(new_node)

    # assign index to node voxels in skel2
    id = len(node) + 1
    skel2.ravel()[new_node.nodes_idx] = id

l_idx = 0

c2n = np.zeros(skel.shape, dtype=int)
c2n_flat = c2n.ravel()
c2n_flat[cans[:, 0]] = np.arange(1, cans.shape[0] + 1)
c2n = c2n_flat

s2n = np.zeros(num_voxels, dtype=int)
s2n[nhi[:, 13]] = np.arange(1, nhi.shape[0] + 1)

for curr_node_cluster_idx in range(len(node)):
    link_indices = s2n[node[curr_node_cluster_idx].nodes_idx] - 1

    for link_idx in link_indices):
        link_cands = nhi[link_indices[link_idx], nh[link_indices[link_idx], :] == 1]

        # short branches with 1 endpoint
        ep_cands = np.intersect1d(link_cands, ep)
        link_cands = link_cands[skel2.ravel()[link_cands] == 1]
        link_cands = np.intersect1d(link_cands, cans[:, 0])

        for can_idx in range(len(link_cands)):
            path, end_node_idx, ep_type = pk_follow_link(skel2, node, curr_node_cluster_idx, link_indices[link_idx], can_idx, cans, c2n)
            print(path)


IndexError: index 4 is out of bounds for axis 0 with size 1

In [ ]:
ep_cands

array([157])

In [ ]:
link_idx = s2n[node[0].nodes_idx] - 1
link_cands = nhi[link_idx[0], nh[link_idx[0], :] == 1]
link_cands

array([ 61,  91,  94, 115])